# Baseline Eval — Qwen2.5 3B (unsloth) + GPT-4o-mini Judge\n\nLoads eval dataset from `aiauyeun/test_dataset` on HuggingFace.\n\nSet your OpenAI key in the config cell below (used only for judging).

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q openai datasets

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."  # paste your key here

JUDGE_MODEL = "gpt-4o-mini"
JUDGE_IN  = 0.15 / 1_000_000
JUDGE_OUT = 0.60 / 1_000_000

RESPONDER_SYSTEM = """You are a Vagaro customer support agent. Vagaro is a business management platform for salons, spas, and fitness businesses.

You will be given retrieved help center chunks and a customer question. Answer using only the provided context.

Rules:
- Be concise and actionable
- Always end your answer with \"Source: <url>\" for every article you draw from
- If the context does not clearly answer the question, say \"I don't have enough information to answer that confidently.\" Do not make things up."""

JUDGE_SYSTEM = """You are evaluating a customer support AI response for correctness.

You will be given:
- A customer question
- A ground truth answer
- The AI's generated response

Mark the response CORRECT (true) if it covers the central idea and would genuinely help the customer answer their question — even if it uses different wording, includes extra detail, or omits minor specifics.

Mark it INCORRECT (false) ONLY if it:
- Contradicts the ground truth on a key fact
- Completely misses the main point of the answer
- Would mislead the customer or send them in the wrong direction

Minor wording differences, extra helpful detail, or slight omissions of non-critical info are NOT a reason to mark false.

Respond in this exact JSON format:
{\"correct\": true/false, \"justification\": \"one sentence\"}"""

In [ ]:
from datasets import load_dataset

ds = load_dataset("aiauyeun/test_dataset", split="train")
all_rows = [dict(r) for r in ds]

# set to None to use all
N_POSITIVE = 5
N_NEGATIVE = 5

pos = [r for r in all_rows if r["question_type"] == "positive"]
neg = [r for r in all_rows if r["question_type"] == "negative"]

if N_POSITIVE is not None:
    pos = pos[:N_POSITIVE]
if N_NEGATIVE is not None:
    neg = neg[:N_NEGATIVE]

rows = pos + neg
print(f"{len(rows)} rows ({len(pos)} positive, {len(neg)} negative)")

In [ ]:
import torch

def generate(question: str, context: str) -> str:
    messages = [
        {"role": "system", "content": RESPONDER_SYSTEM},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


results = []
for i, row in enumerate(rows):
    response = generate(row["question"], row["context"])
    results.append({**row, "generated_response": response})
    print(f"[{i+1}/{len(rows)}] [{row['question_type'].upper()[:3]}] {row['question'][:60]}")

print("\nInference done.")

In [ ]:
import time
import threading
import openai
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed

client = OpenAI()

def with_backoff(fn, *args, start=3.0, max_retries=6, **kwargs):
    wait = start
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except openai.RateLimitError:
            if attempt == max_retries - 1:
                raise
            time.sleep(wait)
            wait *= 2


def judge(row: dict):
    r = with_backoff(
        OpenAI().chat.completions.create,
        model=JUDGE_MODEL,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": f"Question: {row['question']}\n\nGround truth: {row['ground_truth']}\n\nGenerated response: {row['generated_response']}"},
        ],
        response_format={"type": "json_object"},
        temperature=0.0,
    )
    verdict = json.loads(r.choices[0].message.content)
    cost = r.usage.prompt_tokens * JUDGE_IN + r.usage.completion_tokens * JUDGE_OUT
    return {**row, "correct": verdict["correct"], "justification": verdict["justification"]}, cost


judged = [None] * len(results)
total_cost = 0.0
done_count = 0
lock = threading.Lock()

with ThreadPoolExecutor(max_workers=12) as pool:
    futures = {pool.submit(judge, row): i for i, row in enumerate(results)}
    for future in as_completed(futures):
        i = futures[future]
        result, cost = future.result()
        with lock:
            judged[i] = result
            total_cost += cost
            done_count += 1
            status = "OK  " if result["correct"] else "FAIL"
            print(f"[{done_count}/{len(results)}] [{result['question_type'].upper()[:3]}] [{status}] {result['question'][:55]}  (${total_cost:.4f})")

print(f"\nJudging done. Total cost: ${total_cost:.4f}")

In [ ]:
pos = [r for r in judged if r["question_type"] == "positive"]
neg = [r for r in judged if r["question_type"] == "negative"]

pos_acc = sum(r["correct"] for r in pos) / len(pos) if pos else 0
neg_acc = sum(r["correct"] for r in neg) / len(neg) if neg else 0
overall = sum(r["correct"] for r in judged) / len(judged) if judged else 0

print("=" * 45)
print(f"  Positive accuracy : {pos_acc:.1%}  ({sum(r['correct'] for r in pos)}/{len(pos)})")
print(f"  Negative accuracy : {neg_acc:.1%}  ({sum(r['correct'] for r in neg)}/{len(neg)})")
print(f"  Overall accuracy  : {overall:.1%}  ({sum(r['correct'] for r in judged)}/{len(judged)})")
print(f"  Judge cost        : ${total_cost:.4f}")
print("=" * 45)

In [ ]:
import csv

out_path = "judge_results_qwen_baseline.csv"
fieldnames = ["question_type", "question", "ground_truth", "generated_response", "correct", "justification"]
with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(judged)

print(f"Saved to {out_path}")

from google.colab import files
files.download(out_path)